# 04. Publication Exports and Final Checklist

This notebook verifies that all required exports exist and that the numerical quality checks pass. Missing upstream exports are regenerated in notebook order.

In [1]:
from pathlib import Path
import subprocess
import sys

import pandas as pd
from IPython.display import Markdown, display

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "notebooks").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent.resolve()
NOTEBOOK_DIR = PROJECT_ROOT / "notebooks"
FIGURES_DIR = PROJECT_ROOT / "figures"
TABLES_DIR = PROJECT_ROOT / "tables"

def save_dataframe(df: pd.DataFrame, filename: str) -> Path:
    path = TABLES_DIR / filename
    df.to_csv(path, index=False)
    if not path.exists():
        raise FileNotFoundError(f"Failed to save {path}.")
    return path

def execute_notebook(notebook_name: str) -> None:
    notebook_path = NOTEBOOK_DIR / notebook_name
    if not notebook_path.exists():
        raise FileNotFoundError(f"Missing notebook {notebook_path}.")
    command = [
        sys.executable,
        "-m",
        "jupyter",
        "nbconvert",
        "--to",
        "notebook",
        "--execute",
        "--inplace",
        "--ExecutePreprocessor.timeout=3600",
        "--ExecutePreprocessor.kernel_name=codex-qft-split",
        str(notebook_path),
    ]
    subprocess.run(command, cwd=PROJECT_ROOT, check=True)

In [2]:
expected_outputs = [
    ("figures/harmonic_density_snapshots.png", "01_harmonic_oscillator_qft_split_operator.ipynb"),
    ("figures/harmonic_density_snapshots.pdf", "01_harmonic_oscillator_qft_split_operator.ipynb"),
    ("figures/harmonic_fidelity_vs_time.png", "01_harmonic_oscillator_qft_split_operator.ipynb"),
    ("figures/harmonic_fidelity_vs_time.pdf", "01_harmonic_oscillator_qft_split_operator.ipynb"),
    ("figures/harmonic_spatial_convergence.png", "01_harmonic_oscillator_qft_split_operator.ipynb"),
    ("figures/harmonic_spatial_convergence.pdf", "01_harmonic_oscillator_qft_split_operator.ipynb"),
    ("tables/harmonic_parameters.csv", "01_harmonic_oscillator_qft_split_operator.ipynb"),
    ("tables/harmonic_fidelity_vs_time.csv", "01_harmonic_oscillator_qft_split_operator.ipynb"),
    ("tables/harmonic_fidelity_vs_r.csv", "01_harmonic_oscillator_qft_split_operator.ipynb"),
    ("tables/harmonic_spatial_convergence.csv", "01_harmonic_oscillator_qft_split_operator.ipynb"),
    ("figures/infinite_well_density_snapshots.png", "02_infinite_well_qft_split_operator.ipynb"),
    ("figures/infinite_well_density_snapshots.pdf", "02_infinite_well_qft_split_operator.ipynb"),
    ("figures/infinite_well_fidelity_vs_time.png", "02_infinite_well_qft_split_operator.ipynb"),
    ("figures/infinite_well_fidelity_vs_time.pdf", "02_infinite_well_qft_split_operator.ipynb"),
    ("figures/infinite_well_spatial_convergence.png", "02_infinite_well_qft_split_operator.ipynb"),
    ("figures/infinite_well_spatial_convergence.pdf", "02_infinite_well_qft_split_operator.ipynb"),
    ("tables/infinite_well_parameters.csv", "02_infinite_well_qft_split_operator.ipynb"),
    ("tables/infinite_well_fidelity_vs_time.csv", "02_infinite_well_qft_split_operator.ipynb"),
    ("tables/infinite_well_fidelity_vs_r.csv", "02_infinite_well_qft_split_operator.ipynb"),
    ("tables/infinite_well_spatial_convergence.csv", "02_infinite_well_qft_split_operator.ipynb"),
    ("figures/harmonic_single_step_logical_circuit.png", "03_circuit_resource_analysis.ipynb"),
    ("figures/harmonic_single_step_logical_circuit.pdf", "03_circuit_resource_analysis.ipynb"),
    ("figures/harmonic_single_step_transpiled_circuit.png", "03_circuit_resource_analysis.ipynb"),
    ("figures/harmonic_single_step_transpiled_circuit.pdf", "03_circuit_resource_analysis.ipynb"),
    ("figures/infinite_well_single_step_logical_circuit.png", "03_circuit_resource_analysis.ipynb"),
    ("figures/infinite_well_single_step_logical_circuit.pdf", "03_circuit_resource_analysis.ipynb"),
    ("figures/infinite_well_single_step_transpiled_circuit.png", "03_circuit_resource_analysis.ipynb"),
    ("figures/infinite_well_single_step_transpiled_circuit.pdf", "03_circuit_resource_analysis.ipynb"),
    ("figures/harmonic_gate_counts_vs_steps.png", "03_circuit_resource_analysis.ipynb"),
    ("figures/harmonic_gate_counts_vs_steps.pdf", "03_circuit_resource_analysis.ipynb"),
    ("figures/infinite_well_gate_counts_vs_steps.png", "03_circuit_resource_analysis.ipynb"),
    ("figures/infinite_well_gate_counts_vs_steps.pdf", "03_circuit_resource_analysis.ipynb"),
    ("figures/harmonic_fidelity_vs_two_qubit_gate_count.png", "03_circuit_resource_analysis.ipynb"),
    ("figures/harmonic_fidelity_vs_two_qubit_gate_count.pdf", "03_circuit_resource_analysis.ipynb"),
    ("figures/harmonic_fidelity_vs_total_gate_count.png", "03_circuit_resource_analysis.ipynb"),
    ("figures/harmonic_fidelity_vs_total_gate_count.pdf", "03_circuit_resource_analysis.ipynb"),
    ("figures/infinite_well_fidelity_vs_two_qubit_gate_count.png", "03_circuit_resource_analysis.ipynb"),
    ("figures/infinite_well_fidelity_vs_two_qubit_gate_count.pdf", "03_circuit_resource_analysis.ipynb"),
    ("figures/infinite_well_fidelity_vs_total_gate_count.png", "03_circuit_resource_analysis.ipynb"),
    ("figures/infinite_well_fidelity_vs_total_gate_count.pdf", "03_circuit_resource_analysis.ipynb"),
    ("tables/circuit_validation.csv", "03_circuit_resource_analysis.ipynb"),
    ("tables/circuit_single_step_metrics.csv", "03_circuit_resource_analysis.ipynb"),
    ("tables/circuit_gate_breakdown.csv", "03_circuit_resource_analysis.ipynb"),
    ("tables/resource_totals_vs_r.csv", "03_circuit_resource_analysis.ipynb"),
]

In [3]:
missing_by_notebook = {}
for relative_path, source_notebook in expected_outputs:
    path = PROJECT_ROOT / relative_path
    if not path.exists():
        missing_by_notebook.setdefault(source_notebook, []).append(path.name)

if missing_by_notebook:
    display(Markdown("## Regenerating Missing Exports"))
    for notebook_name in [
        "01_harmonic_oscillator_qft_split_operator.ipynb",
        "02_infinite_well_qft_split_operator.ipynb",
        "03_circuit_resource_analysis.ipynb",
    ]:
        if notebook_name in missing_by_notebook:
            print(f"Re-executing {notebook_name} because required exports were missing.")
            execute_notebook(notebook_name)

manifest_df = pd.DataFrame(
    [
        {
            "path": relative_path,
            "exists": (PROJECT_ROOT / relative_path).exists(),
            "source_notebook": source_notebook,
        }
        for relative_path, source_notebook in expected_outputs
    ]
)
if not manifest_df["exists"].all():
    raise FileNotFoundError("At least one required publication export is still missing after regeneration.")

save_dataframe(manifest_df, "publication_manifest.csv")
display(Markdown("## Publication Manifest"))
display(manifest_df)

## Publication Manifest

,path,exists,source_notebook
0,figures/harmonic_density_snapshots.png,True,01_harmonic_oscillator_qft_split_operator.ipynb
1,figures/harmonic_density_snapshots.pdf,True,01_harmonic_oscillator_qft_split_operator.ipynb
2,figures/harmonic_fidelity_vs_time.png,True,01_harmonic_oscillator_qft_split_operator.ipynb
3,figures/harmonic_fidelity_vs_time.pdf,True,01_harmonic_oscillator_qft_split_operator.ipynb
4,figures/harmonic_spatial_convergence.png,True,01_harmonic_oscillator_qft_split_operator.ipynb
5,figures/harmonic_spatial_convergence.pdf,True,01_harmonic_oscillator_qft_split_operator.ipynb
6,tables/harmonic_parameters.csv,True,01_harmonic_oscillator_qft_split_operator.ipynb
7,tables/harmonic_fidelity_vs_time.csv,True,01_harmonic_oscillator_qft_split_operator.ipynb
8,tables/harmonic_fidelity_vs_r.csv,True,01_harmonic_oscillator_qft_split_operator.ipynb
9,tables/harmonic_spatial_convergence.csv,True,01_harmonic_oscillator_qft_split_operator.ipynb


In [4]:
harmonic_params = pd.read_csv(TABLES_DIR / "harmonic_parameters.csv")
well_params = pd.read_csv(TABLES_DIR / "infinite_well_parameters.csv")
harmonic_fidelity = pd.read_csv(TABLES_DIR / "harmonic_fidelity_vs_time.csv")
well_fidelity = pd.read_csv(TABLES_DIR / "infinite_well_fidelity_vs_time.csv")
validation = pd.read_csv(TABLES_DIR / "circuit_validation.csv")
resource_totals = pd.read_csv(TABLES_DIR / "resource_totals_vs_r.csv")

quality_records = [
    {"check": "Harmonic final fidelity above 0.999", "status": float(harmonic_fidelity["fidelity"].iloc[-1]) > 0.999},
    {"check": "Harmonic minimum fidelity above 0.999", "status": float(harmonic_fidelity["fidelity"].min()) > 0.999},
    {"check": "Harmonic edge probability below 1e-4", "status": float(harmonic_params["max_edge_probability"].iloc[0]) < 1e-4},
    {"check": "Infinite-well final fidelity above 0.999", "status": float(well_fidelity["fidelity"].iloc[-1]) > 0.999},
    {"check": "Infinite-well minimum fidelity above 0.999", "status": float(well_fidelity["fidelity"].min()) > 0.999},
    {"check": "Infinite-well periodic mismatch removed", "status": bool(well_params["periodic_boundary_mismatch"].iloc[0]) is False},
    {"check": "Transform validations passed", "status": bool(validation["passed"].all())},
    {"check": "Resource table records synthesis model", "status": "synthesis_model" in resource_totals.columns},
]
quality_df = pd.DataFrame(quality_records)
display(Markdown("## Numerical Quality Checks"))
display(quality_df)

if not quality_df["status"].all():
    raise RuntimeError("At least one numerical quality check failed.")

## Numerical Quality Checks

,check,status
0,Harmonic final fidelity above 0.999,True
1,Harmonic minimum fidelity above 0.999,True
2,Harmonic edge probability below 1e-4,True
3,Infinite-well final fidelity above 0.999,True
4,Infinite-well minimum fidelity above 0.999,True
5,Infinite-well periodic mismatch removed,True
6,Transform validations passed,True
7,Resource table records synthesis model,True


In [5]:
summary_md = '''## Concise Methods Summary

**Harmonic oscillator:** second-order symmetric split-operator evolution on a periodic 64-point QFT grid, benchmarked against a truncated analytical harmonic-oscillator eigenbasis and accompanied by boundary and spatial-convergence diagnostics.

**Infinite well:** corrected hard-wall simulation using an orthonormal Dirichlet sine transform, the boundary-compatible Fourier/QFT-family transform. This removes the earlier periodic-ring mismatch and is benchmarked against the analytical sine-basis reference.

**Circuit resources:** the harmonic oscillator uses a validated Qiskit QFT convention and exact small-N generic diagonal synthesis. The infinite well reports an analytical quantum-sine-transform resource estimate, explicitly separated from the harmonic generic-synthesis counts.
'''
display(Markdown(summary_md))
display(Markdown("## Key Tables"))
display(harmonic_params)
display(well_params)
display(pd.read_csv(TABLES_DIR / "circuit_single_step_metrics.csv"))
display(resource_totals.head())

## Concise Methods Summary

**Harmonic oscillator:** second-order symmetric split-operator evolution on a periodic 64-point QFT grid, benchmarked against a truncated analytical harmonic-oscillator eigenbasis and accompanied by boundary and spatial-convergence diagnostics.

**Infinite well:** corrected hard-wall simulation using an orthonormal Dirichlet sine transform, the boundary-compatible Fourier/QFT-family transform. This removes the earlier periodic-ring mismatch and is benchmarked against the analytical sine-basis reference.

**Circuit resources:** the harmonic oscillator uses a validated Qiskit QFT convention and exact small-N generic diagonal synthesis. The infinite well reports an analytical quantum-sine-transform resource estimate, explicitly separated from the harmonic generic-synthesis counts.


## Key Tables

,system,numerical_transform,boundary_model,N,n_qubits,domain,x0,k0,sigma,t_max,...,hbar,m,omega,reference_grid_size,reference_basis_cap,reference_basis_kept,reference_tail_weight,max_edge_probability,fidelity_sweep_r,spatial_convergence_N
0,harmonic_oscillator,periodic_QFT_or_FFT,finite_periodic_box_checked_by_edge_probability,64,6,"[-8.0, 8.0)",2.0,0.0,1.0,6.283185,...,1.0,1.0,1.0,4096,128,64,9.371548e-11,5.084142e-07,"50,100,150,200,250,300,400","32,64,128"


,system,numerical_transform,boundary_model,periodic_boundary_mismatch,N,n_qubits,domain,L,x0,k0,...,hbar,m,reference_grid_size,reference_basis_cap,reference_basis_kept,reference_tail_weight,initial_state_modifier,boundary_caveat,fidelity_sweep_r,spatial_convergence_N
0,infinite_well,orthonormal_DST-II_quantum_sine_transform_family,Dirichlet_hard_wall,False,64,6,"(0, 10.0) midpoint grid",10.0,5.0,2.0,...,1.0,1.0,4097,256,20,4.473977e-12,sin(pi x / L),resolved_by_Dirichlet_sine_transform_not_perio...,"100,200,300,400,500,600,700","32,64,128"


,system,N,n_qubits,dt,single_step_1q_count,single_step_2q_count,single_step_depth,transform,synthesis_model
0,harmonic_oscillator,64,6,0.031416,171,264,312,periodic_QFT,Qiskit_exact_generic_DiagonalGate_small_N_uppe...
1,infinite_well,64,6,0.008571,235,142,188,Dirichlet_quantum_sine_transform,analytical_all_to_all_QST_via_QFT_extension_es...


,system,r,dt,final_time_fidelity,minimum_fidelity,mean_fidelity,single_step_1q_count,single_step_2q_count,single_step_depth,synthesis_model,total_1q_count,total_2q_count,total_gate_count,estimated_total_depth
0,harmonic_oscillator,50,0.125664,0.999927,0.999905,0.999968,171,264,312,Qiskit_exact_generic_DiagonalGate_small_N_uppe...,8550,13200,21750,15600
1,harmonic_oscillator,100,0.062832,0.999995,0.999994,0.999998,171,264,312,Qiskit_exact_generic_DiagonalGate_small_N_uppe...,17100,26400,43500,31200
2,harmonic_oscillator,150,0.041888,0.999999,0.999999,1.000000,171,264,312,Qiskit_exact_generic_DiagonalGate_small_N_uppe...,25650,39600,65250,46800
3,harmonic_oscillator,200,0.031416,1.000000,1.000000,1.000000,171,264,312,Qiskit_exact_generic_DiagonalGate_small_N_uppe...,34200,52800,87000,62400
4,harmonic_oscillator,250,0.025133,1.000000,1.000000,1.000000,171,264,312,Qiskit_exact_generic_DiagonalGate_small_N_uppe...,42750,66000,108750,78000


In [6]:
checklist_df = pd.DataFrame(
    [
        {"check": "Environment notebook present", "status": (NOTEBOOK_DIR / "00_environment_setup.ipynb").exists()},
        {"check": "Harmonic dynamics notebook present", "status": (NOTEBOOK_DIR / "01_harmonic_oscillator_qft_split_operator.ipynb").exists()},
        {"check": "Infinite-well dynamics notebook present", "status": (NOTEBOOK_DIR / "02_infinite_well_qft_split_operator.ipynb").exists()},
        {"check": "Circuit notebook present", "status": (NOTEBOOK_DIR / "03_circuit_resource_analysis.ipynb").exists()},
        {"check": "Publication notebook present", "status": (NOTEBOOK_DIR / "04_publication_exports.ipynb").exists()},
        {"check": "All required figures and tables exist", "status": bool(manifest_df["exists"].all())},
        {"check": "CSV manifest saved", "status": (TABLES_DIR / "publication_manifest.csv").exists()},
        {"check": "All numerical quality checks pass", "status": bool(quality_df["status"].all())},
        {"check": "Boundary model corrected for infinite well", "status": bool(well_params["periodic_boundary_mismatch"].iloc[0]) is False},
    ]
)
save_dataframe(checklist_df, "final_checklist.csv")
display(Markdown("## Final Checklist"))
display(checklist_df)

if not checklist_df["status"].all():
    raise RuntimeError("The final checklist contains at least one failed item.")
print("Publication export check completed successfully.")

## Final Checklist

,check,status
0,Environment notebook present,True
1,Harmonic dynamics notebook present,True
2,Infinite-well dynamics notebook present,True
3,Circuit notebook present,True
4,Publication notebook present,True
5,All required figures and tables exist,True
6,CSV manifest saved,True
7,All numerical quality checks pass,True
8,Boundary model corrected for infinite well,True


Publication export check completed successfully.
